# Obtención de datos generales.

Se obtienen datos generales sobre el nivel de vida por departamento y de puntos de interés de la propiedad a evaluar. 

Requerimientos:

- Base de EHPM actualizada (.sav)
- Coordinadas de la(s) propiedad(es) a evaluar (en excel, cada propiedad representa una fila)


Output:

- Base con los atributos necesarios para calcular la predicción del precio.



## 1. Atributos de Nivel de vida por departamento

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

### Depuración de datos

In [ ]:
df = pd.read_spss("EHPM 2021.sav") # Se debe sustituir por el nombre del archivo.

In [ ]:
print(df['r020'].value_counts(normalize = True))

## Se eliminan los registros con información poco o no confiable (aprox. 1.2%)

df = df[df['r020']=='Información confiable']
df.shape

Información confiable        0.99
Informacion poco confiable   0.01
Información no confiable     0.00
Name: r020, dtype: float64


(64230, 648)

In [ ]:
## Se considera hogares con al menos un 80% de las preguntas contestadas
df_depurado = df[variables].dropna(thresh=df[variables].shape[1]*0.8,axis=0)
df_depurado.shape

(64133, 163)

### Cálculo de atributos a nivel departamento


In [ ]:
df_depurado['porc_hab'] = np.where(df_depurado['r305']!=0, df_depurado['r306']/df_depurado['r305'],np.nan)
calculo_attrs1 = df_depurado.groupby('r004').agg({'r021a': 'mean'}).reset_index()
calculo_attrs1.columns = ['DEPTO','miembros_ph_avg']


In [ ]:
# Normalizar valores
df_depurado['hogares_lectura'] = np.where(df_depurado['r202a'].isin(['Sí', 'Sí, sólo leer']),1,0)
df_depurado['hogares_techo_lamina'] = np.where(df_depurado['r302'].isin(['Lámina metálica']),1,0)
df_depurado['hogares_sin_agua'] = np.where(df_depurado['r312'].isin(['No tiene' ]),1,0)

crear = ['hogares_lectura','hogares_techo_lamina','hogares_sin_agua']

In [ ]:
atributos_depto = df_depurado.groupby('r004').size().reset_index()
atributos_depto.columns = ['DEPTO','CONTEO']
atributos_depto

In [225]:
for vari in crear:
    conteo = pd.crosstab(index = df_depurado['r004'], columns = df_depurado[vari], normalize = 'index').reset_index()
    conteo.columns = ['DEPTO','NO',vari]
    conteo = conteo[['DEPTO',vari]]
    atributos_depto = atributos_depto.merge(conteo, on = 'DEPTO')
    

In [227]:
atributos_depto = atributos_depto.merge(calculo_attrs1, on = 'DEPTO')

In [ ]:
# Guardar resultados. Opcional, si solo se desea calcular estos atributos el código termina aquí.

atributos_depto.to_csv('atributos_departamento.csv')

## 2. Puntos de interés cercanos a la propiedad

In [ ]:

import requests
import time

In [ ]:
# === CONFIGURACIÓN ===
API_KEY = ''  # 🔑 Reemplaza con tu clave de Google Maps
RADIUS = 3000           # Radio en metros (máx. 50,000)
place_types = ["university", "school"] 

In [ ]:
# El Excel debe tener columnas llamadas 'latitud' y 'longitud'.
# El nombre del archivo se debe sustituir por el que el usuario posea
df_coords = pd.read_excel("Centroides de Distritos a Utilizar.xlsx", sheet_name="Centroides y Cuadricula por Dis")

In [ ]:
# === 2. FUNCIÓN PARA CONSULTAR LA API ===
# Función para consultar la API (nuevo formato)
def search_places(lat, lon, place_type, radius=3000):
    url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.types"
    }

    body = {
        "includedTypes": [place_type],
        "maxResultCount": 10,
        "locationRestriction": {
            "circle": {
                "center": {"latitude": lat, "longitude": lon},
                "radius": radius
            }
        }
    }   

    res = requests.post(url, headers=headers, json=body)
    data = res.json()
    return data

In [ ]:
# Recolectar resultados
results = []

for _, row in df_coords.iterrows():
    lat = row["lat"]
    lon = row["lon"]
    for place_type in place_types:
        data = search_places(lat, lon, place_type)
        if "places" in data:
            for p in data["places"]:
                results.append({
                    "latitud": lat,
                    "longitud": lon,
                    "tipo": place_type,
                    "nombre": p.get("displayName", {}).get("text", ""),
                    "direccion": p.get("formattedAddress", ""),
                    "lat": p.get("location", {}).get("latitude", None),
                    "lon": p.get("location", {}).get("longitude", None)
                })
        else:
            print(f"Sin resultados para {place_type} en ({lat}, {lon})")
    time.sleep(1)  # para no saturar el límite de solicitudes



In [ ]:
# Guardar resultados
pd.DataFrame(results).to_csv("POIS.csv", index=False)